In [1]:
#| echo: false
#| warning: false
#| output: asis

from IPython.display import Markdown, display

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from statsmodels.formula.api import ols
from statsmodels.iolib.summary2 import summary_params, summary_col
from scipy.stats import t, f, norm

# Не показывать Warning
import warnings
warnings.simplefilter(action='ignore', category=Warning)
# Не показывать ValueWarning, ConvergenceWarning из statsmodels
from statsmodels.tools.sm_exceptions import ValueWarning, ConvergenceWarning

# Загрузка данных
loanapp = pd.read_csv('../datasets/loanapp.csv')
mroz_Greene = pd.read_csv('../datasets/TableF5-1.csv')

loanapp = loanapp[ ['appinc', 'mortno', 'unem', 'dep', 'male', 'married', 'yjob', 'self', 'approve'] ].dropna()

# Настройки оформления графиков
plt.rcParams['figure.figsize'] = (8.5, 6)
plt.rcParams['font.size'] = 10
plt.rcParams['axes.labelsize'] = 12
plt.rcParams['axes.titlesize'] = 14

# Подгонка модели

## approve equation #1

Для датасета `loanapp` рассмотрим регрессию __approve на mortno, unem, dep, male, married, yjob, self__

Спецификация:

$$
\begin{equation}
approve=\beta_0 + \beta_1 mortno+\beta_2 unem + \beta_3 dep + \beta_4 male + \beta_5 married + \beta_6 yjob + \beta_7 self + u
\end{equation}
$$

Альтернативная спецификация:

$$
\begin{equation}
P(approve=1)=\beta_0+\beta_1mortno+\beta_2unem+\beta_3dep+\beta_4male+\beta_5married+\beta_6yjob+\beta_7self
\end{equation}
$$

Оцените модель на данных и укажите коэффициенты подогнанной модели. __Ответ округлите до 3-х десятичных знаков.__

Ответ:

In [2]:
#| echo: false
#| warning: false
#| output: asis

res = ols(formula = 'approve ~ mortno + unem + dep + male + married + yjob + self', data=loanapp).fit()
res_hc = ols(formula = 'approve ~ mortno + unem + dep + male + married + yjob + self', data=loanapp).fit(cov_type='HC3')

digits  = 3
# Формирование таблицы результатов
results_df = pd.DataFrame({
    'Переменная': res.params.index,
    'Оценка (OLS)': np.round(res.params, digits),
    #'Стд. ошибка (OLS)': np.round(res.bse, digits),
    'Оценка (HC3)': np.round(res_hc.params, digits),
    #'Стд. ошибка (HC3)': np.round(res_hc.bse, digits)
})

print(results_df.to_markdown(index=False))

| Переменная   |   Оценка (OLS) |   Оценка (HC3) |
|:-------------|---------------:|---------------:|
| Intercept    |          0.864 |          0.864 |
| mortno       |          0.073 |          0.073 |
| unem         |         -0.006 |         -0.006 |
| dep          |         -0.018 |         -0.018 |
| male         |          0.002 |          0.002 |
| married      |          0.046 |          0.046 |
| yjob         |         -0.001 |         -0.001 |
| self         |         -0.036 |         -0.036 |


__Дайте интерпретацию коэффициентам модели.__

## approve equation #2

Для датасета `loanapp` рассмотрим регрессию __approve на appinc, I(appinc ** 2), mortno, unem, dep, male, married, yjob, self__

Спецификация:

$$
\begin{equation}
approve=\beta_0 + \beta_1 appinc + \beta_2 appinc^{2} + \beta_3 mortno+\beta_4 unem + \beta_5 dep + \beta_6 male + \beta_7 married + \beta_8 yjob + \beta_9 self + u
\end{equation}
$$

Альтернативная спецификация:

$$
\begin{equation}
P(approve=1)=\beta_0 + \beta_1 appinc + \beta_2 appinc^{2} + \beta_3 mortno+\beta_4 unem + \beta_5 dep + \beta_6 male + \beta_7 married + \beta_8 yjob + \beta_9 self
\end{equation}
$$


Оцените модель на данных и укажите коэффициенты подогнанной модели. __Ответ округлите до 3-х десятичных знаков.__

Ответ:

In [3]:
#| echo: false
#| warning: false
#| output: asis

res = ols(formula = 'approve ~ appinc + I(appinc ** 2) + mortno + unem + dep + male + married + yjob + self', data=loanapp).fit()
res_hc = ols(formula = 'approve ~ appinc + I(appinc ** 2) + mortno + unem + dep + male + married + yjob + self', data=loanapp).fit(cov_type='HC3')

digits  = 3
# Формирование таблицы результатов
results_df = pd.DataFrame({
    'Переменная': res.params.index,
    'Оценка (OLS)': np.round(res.params, digits),
    #'Стд. ошибка (OLS)': np.round(res.bse, digits),
    'Оценка (HC3)': np.round(res_hc.params, digits),
    #'Стд. ошибка (HC3)': np.round(res_hc.bse, digits)
})

print(results_df.to_markdown(index=False))

| Переменная     |   Оценка (OLS) |   Оценка (HC3) |
|:---------------|---------------:|---------------:|
| Intercept      |          0.842 |          0.842 |
| appinc         |          0.001 |          0.001 |
| I(appinc ** 2) |         -0     |         -0     |
| mortno         |          0.066 |          0.066 |
| unem           |         -0.006 |         -0.006 |
| dep            |         -0.017 |         -0.017 |
| male           |         -0.003 |         -0.003 |
| married        |          0.043 |          0.043 |
| yjob           |         -0.001 |         -0.001 |
| self           |         -0.04  |         -0.04  |


__Дайте интерпретацию коэффициентам модели.__

## labour force equation #1

Для датасета `TableF5-1` рассмотрим регрессию __LFP на WA, I(WA ** 2), WE, KL6, K618, CIT, UN, np.log(FAMINC)__

Спецификация:

$$
\begin{equation}
LFP=\beta_0+\beta_1WA+\beta_2WA^2+\beta_3WE+\beta_4KL6+\beta_5K618+\beta_6CIT+\beta_7UN+\beta_8\log(FAMINC)+u
\end{equation}
$$

Альтернативная спецификация:

$$
\begin{equation}
P(LFP=1)=\beta_0+\beta_1WA+\beta_2WA^2+\beta_3WE+\beta_4KL6+\beta_5K618+\beta_6CIT+\beta_7UN+\beta_8\log(FAMINC)
\end{equation}
$$

Оцените модель на данных и укажите коэффициенты подогнанной модели. __Ответ округлите до 3-х десятичных знаков.__

Ответ:

In [4]:
#| echo: false
#| warning: false
#| output: asis

res = ols(formula = 'LFP ~ WA + I(WA ** 2) + WE + KL6 + K618 + CIT + UN + np.log(FAMINC)', data=mroz_Greene).fit()
res_hc = ols(formula = 'LFP ~ WA + I(WA ** 2) + WE + KL6 + K618 + CIT + UN + np.log(FAMINC)', data=mroz_Greene).fit(cov_type='HC3')

digits  = 3
# Формирование таблицы результатов
results_df = pd.DataFrame({
    'Переменная': res.params.index,
    'Оценка (OLS)': np.round(res.params, digits),
    #'Стд. ошибка (OLS)': np.round(res.bse, digits),
    'Оценка (HC3)': np.round(res_hc.params, digits),
    #'Стд. ошибка (HC3)': np.round(res_hc.bse, digits)
})

print(results_df.to_markdown(index=False))

| Переменная     |   Оценка (OLS) |   Оценка (HC3) |
|:---------------|---------------:|---------------:|
| Intercept      |         -0.321 |         -0.321 |
| WA             |          0.008 |          0.008 |
| I(WA ** 2)     |         -0     |         -0     |
| WE             |          0.038 |          0.038 |
| KL6            |         -0.296 |         -0.296 |
| K618           |         -0.021 |         -0.021 |
| CIT            |         -0.048 |         -0.048 |
| UN             |         -0.004 |         -0.004 |
| np.log(FAMINC) |          0.072 |          0.072 |


__Дайте интерпретацию коэффициентам модели.__

## labour force equation #2

Для датасета `TableF5-1` рассмотрим регрессию __LFP на WA, WE, CIT, UN, np.log(FAMINC)__

Спецификация:

$$
\begin{equation}
LFP = \beta_0 + \beta_1 WA + \beta_2 WE + \beta_3 CIT + \beta_4 CIT+\beta_5 UN + \beta_6 \log(FAMINC)+u
\end{equation}
$$

Альтернативная спецификация:

$$
\begin{equation}
P(LFP=1) = \beta_0 + \beta_1 WA + \beta_2 WE + \beta_3 CIT + \beta_4 CIT+\beta_5 UN + \beta_6 \log(FAMINC)
\end{equation}
$$

Оцените модель на данных и укажите коэффициенты подогнанной модели. __Ответ округлите до 3-х десятичных знаков.__

Ответ:

In [5]:
#| echo: false
#| warning: false
#| output: asis

res = ols(formula = 'LFP ~ WA + WE + CIT + UN + np.log(FAMINC)', data=mroz_Greene).fit()
res_hc = ols(formula = 'LFP ~ WA + WE + CIT + UN + np.log(FAMINC)', data=mroz_Greene).fit(cov_type='HC3')

digits  = 3
# Формирование таблицы результатов
results_df = pd.DataFrame({
    'Переменная': res.params.index,
    'Оценка (OLS)': np.round(res.params, digits),
    #'Стд. ошибка (OLS)': np.round(res.bse, digits),
    'Оценка (HC3)': np.round(res_hc.params, digits),
    #'Стд. ошибка (HC3)': np.round(res_hc.bse, digits)
})

print(results_df.to_markdown(index=False))

| Переменная     |   Оценка (OLS) |   Оценка (HC3) |
|:---------------|---------------:|---------------:|
| Intercept      |         -0.536 |         -0.536 |
| WA             |         -0.004 |         -0.004 |
| WE             |          0.033 |          0.033 |
| CIT            |         -0.048 |         -0.048 |
| UN             |         -0.005 |         -0.005 |
| np.log(FAMINC) |          0.094 |          0.094 |


__Дайте интерпретацию коэффициентам модели.__

## Замечание: почему log(FAMINC)

Нарисуем гистограмму $\textrm{FAMINC}$ с наложенной кривой нормального распределения

In [6]:
#| echo: false
#| warning: false
#| label: fig-faminc-dist
#| fig-cap: "Гистограмма распределения дохода семьи ($\\textrm{FAMINC}$) с наложением нормальной кривой плотности."
#| fig-align: "center"

# Включаем красивый стиль seaborn (можно вынести в самую первую ячейку ноутбука)
#sns.set_theme(style="whitegrid", palette="muted")

FAMINC = mroz_Greene['FAMINC']

#plt.figure(figsize=(8, 5))

# Строим гистограмму с помощью seaborn
sns.histplot(
    FAMINC, 
    bins='fd', 
    stat='density', # Аналог density=True в matplotlib
    alpha=0.6, 
    edgecolor='black',
    label='Эмпирическое распределение'
)

# Наложение теоретического нормального распределения
x_vals = np.linspace(FAMINC.min(), FAMINC.max(), 100)
y_vals = norm.pdf(x_vals, FAMINC.mean(), FAMINC.std())
plt.plot(x_vals, y_vals, 'r-', linewidth=2, label='Кривая плотности нормального распределения')

#plt.title('Распределение дохода семьи (FAMINC)', fontsize=14)
plt.xlabel('FAMINC', fontsize=12)
plt.ylabel('Плотность вероятности', fontsize=12)
plt.legend(fontsize=11)
plt.tight_layout()
plt.show()

<Figure size 2550x1800 with 1 Axes>

Нарисуем гистограмму $\log(\textrm{FAMINC})$ с наложенной кривой нормального распределения

In [7]:
#| echo: false
#| warning: false
#| label: fig-log-faminc
#| fig-cap: "Гистограмма логарифмированного дохода семьи с наложением нормальной кривой плотности."
#| fig-align: "center"

# Логарифмируем данные
log_FAMINC = np.log(mroz_Greene['FAMINC'])

#plt.figure(figsize=(8, 5))

# Гистограмма через seaborn
sns.histplot(
    log_FAMINC, 
    bins='fd', 
    stat='density', 
    alpha=0.6, 
    edgecolor='black',
    label='Эмпирическое распределение'
)

# Наложение теоретического нормального распределения
x_vals = np.linspace(log_FAMINC.min(), log_FAMINC.max(), 100)
y_vals = norm.pdf(x_vals, log_FAMINC.mean(), log_FAMINC.std())
plt.plot(x_vals, y_vals, 'r-', linewidth=2, label='Кривая плотности нормального распределения')

#plt.title('Распределение логарифма дохода семьи', fontsize=14)
plt.xlabel('$log(FAMINC)$', fontsize=12)
plt.ylabel('Плотность вероятности', fontsize=12)
plt.legend(fontsize=11)
plt.tight_layout()
plt.show()

<Figure size 2550x1800 with 1 Axes>

# t-тест

## approve equation #1

Для датасета `loanapp` рассмотрим регрессию __approve на mortno, unem, dep, male, married, yjob, self__

Оцените модель на данных и укажите (робастные) коэффициенты подогнанной модели. __Ответ округлите до 3-х десятичных знаков.__

Ответ:

In [8]:
#| echo: false
#| warning: false
#| output: asis

res = ols(formula = 'approve ~ 1 + mortno + unem + dep + male + married + yjob + self', data=loanapp).fit()
res_hc = ols(formula = 'approve ~ 1 + mortno + unem + dep + male + married + yjob + self', data=loanapp).fit(cov_type='HC3')

digits = 3
# Формирование таблицы результатов
results_df = pd.DataFrame({
    'Переменная': res.params.index,
    #'Оценка (OLS)': np.round(res.params, digits),
    #'Стд. ошибка (OLS)': np.round(res.bse, digits),
    'Оценка (HC3)': np.round(res_hc.params, digits),
    #'Стд. ошибка (HC3)': np.round(res_hc.bse, digits)
})

print(results_df.to_markdown(index=False))

| Переменная   |   Оценка (HC3) |
|:-------------|---------------:|
| Intercept    |          0.864 |
| mortno       |          0.073 |
| unem         |         -0.006 |
| dep          |         -0.018 |
| male         |          0.002 |
| married      |          0.046 |
| yjob         |         -0.001 |
| self         |         -0.036 |


In [9]:
#| echo: false
#| warning: false
#| output: asis

# Уровень значимости
sign_level = 0.01

text = f"""
Модель была подогнана по {res.nobs:.0f} наблюдению. <span style="color: blue">Уровень значимости {sign_level * 100:.0f}%</span>
"""

display(Markdown(text))


Модель была подогнана по 1971 наблюдению. <span style="color: blue">Уровень значимости 1%</span>


Вычислите критическое значения для $t$-теста. __Ответ округлите до 3-х десятичных знаков.__

In [10]:
#| echo: false
#| warning: false
#| output: asis

from IPython.display import display, Markdown
import scipy.stats as stats

def display_answer(text, use_callout=True, theme="note", title="Критическое значение"):
    """
    Универсальная функция вывода текстовых блоков для Quarto.
    
    Параметры:
    - text: текст для вывода (строка).
    - use_callout: True оборачивает в рамку, False выводит как текст.
    - theme: цвет рамки ('tip' - зеленый, 'note' - синий, 'warning' - оранжевый).
    - title: текст заголовка. Если передать None или пустую строку, заголовок скроется.
    """
    
    if use_callout:
        # Формируем блок callout (обязательны пустые строки \n\n по краям)
        if title:
            # С заголовком
            output_md = f"\n\n::: {{.callout-{theme} title=\"{title}\" icon=\"true\"}}\n{text}\n:::\n\n"
        else:
            # Без заголовка (Quarto поддерживает скрытие заголовка, если его не указать)
            output_md = f"\n\n::: {{.callout-{theme} icon=\"true\"}}\n{text}\n:::\n\n"
            
    else:
        # Формируем обычный текст
        if title:
            output_md = f"\n\n**{title}:** {text}\n\n"
        else:
            output_md = f"\n\n{text}\n\n"

    # Печатаем результат
    display(Markdown(output_md))

# --- ВАШИ ВЫЧИСЛЕНИЯ ---

sign_level = 0.01
t_crit = stats.t.isf(q=sign_level/2, df=res.df_resid) 
answer_text = f"$t_{{crit}} = {t_crit:.3f}$" #f"$t_{{crit}} = {t_crit:.3f}$ на уровне значимости $\\alpha = {sign_level * 100:.0f}\%$."


# ================= ПРИМЕРЫ ВЫЗОВА =================

# 1. Зеленая рамка со стандартным заголовком "Ответ"
display_answer(answer_text)

# 2. Обычный текст (Без рамки, но с жирным "Ответ:")
# display_answer(answer_text, use_callout=False)

# 3. Синяя рамка (note) со своим заголовком
# display_answer(answer_text, theme="note", title="Критическое значение t-теста")

# 4. Обычный текст ВООБЩЕ БЕЗ заголовка (просто голый ответ)
# display_answer(answer_text, use_callout=False, title=None)

# 5. Рамка-предупреждение (оранжевая) без заголовка
# display_answer("Осторожно, гетероскедастичность!", theme="warning", title=None)



::: {.callout-note title="Критическое значение" icon="true"}
$t_{crit} = 2.578$
:::



__Укажите результаты робастного и неробастного $t$-теста. Ответ округлите до 3-х десятичных знаков.__

In [11]:
#| echo: false
#| warning: false
#| output: asis

from IPython.display import display, Markdown
from statsmodels.iolib.summary2 import summary_params

def print_regression_summary(res, sign_level=0.05, decimals=3, show_signif=True, show_text=True, use_callout=False, callout_title="Ответ"):
    """
    Форматирует и выводит компактную таблицу t-теста коэффициентов регрессии для Quarto.
    """
    # --- 0. ИСПРАВЛЕНИЕ ШИРИНЫ ТАБЛИЦЫ ДЛЯ HTML ---
    # Отключаем растягивание на 100% и центрируем таблицу. 
    # # При экспорте в PDF этот блок будет проигнорирован.
    # css_fix = "<style>table { width: auto !important; margin-left: auto; margin-right: auto; }</style>\n\n"

    # --- 1. АВТОМАТИЧЕСКИЙ ЗАГОЛОВОК И ТИП ОШИБОК ---
    cov_type = getattr(res, 'cov_type', 'nonrobust')
    
    if cov_type == 'nonrobust':
        title = "__Результаты $t$-теста для коэффициентов (неробастные стандартные ошибки, s.e.)__" #"### t test of coefficients (OLS Standard Errors)"
        error_text = "по классическим стандартным ошибкам"
    else:
        title = f"__Результаты $t$-теста для коэффициентов (робастные стандартные ошибки, {cov_type}-s.e.): __" #"### t test of coefficients (Robust Standard Errors: {cov_type})"
        error_text = f"по робастным стандартным ошибкам {cov_type}"

    # --- 2. ПОДГОТОВКА ТАБЛИЦЫ И ПЕРЕИМЕНОВАНИЕ СТОЛБЦОВ ---
    df_params = summary_params(res, alpha=sign_level).iloc[:, :-2]
    
    df_params = df_params.rename(columns={
        'P>|t|': 'p-value', 
        'P>|z|': 'p-value',
        't': 't-value',
        'z': 'z-value'
    })
    
    legend_text = ""
    
    # --- 3. КОДЫ ЗНАЧИМОСТИ ---
    if show_signif:
        def get_stars(p_value):
            if p_value < 0.01: return '***'
            elif p_value < 0.05: return '**'
            elif p_value < 0.1: return '*'
            return ''
            
        df_params['Signif.'] = df_params.iloc[:, 3].apply(get_stars)
        
        for const_name in ['const', 'Intercept']:
            if const_name in df_params.index:
                df_params.loc[const_name, 'Signif.'] = ''
                
        # Надежный формат: текст курсивом, а сами спецсимволы обернуты в код (обратные кавычки)
        legend_text = "\n*Signif. codes:* `***` $p<0.01$, `**` $p<0.05$, `*` $p<0.1$"
        
    # --- 4. ОКРУГЛЕНИЕ ---
    df_params = df_params.round(decimals)
    
    markdown_table = df_params.to_markdown(index=True)

    # --- 5. ТЕКСТОВЫЙ ВЫВОД ЗНАЧИМЫХ ФАКТОРОВ ---
    summary_text = ""
    if show_text:
        all_sign_coeffs = res.params.index[res.pvalues < sign_level].tolist()
        sign_coeffs = [coef for coef in all_sign_coeffs if coef not in ['const', 'Intercept']]
        percent_level = f"{sign_level * 100:g}"
        
        # Сначала формируем чистый текст
        if sign_coeffs:
            formatted_coeffs = ", ".join([f"`{coef}`" for coef in sign_coeffs])
            raw_text = f"**Коэффициенты, значимые на уровне {percent_level}% ({error_text}):** {formatted_coeffs}."
        else:
            raw_text = f"**На уровне значимости {percent_level}% ({error_text}) значимых коэффициентов не выявлено.**"

        # Если запрошен callout, оборачиваем только текст в блок .callout-note
        if use_callout:
            # Для f-строк фигурные скобки у классов экранируются удвоением: {{.callout-note}}
            summary_text = f"\n\n::: {{.callout-note title=\"{callout_title}\" icon=\"true\"}}\n{raw_text}\n:::"
        else:
            summary_text = f"\n\n{raw_text}"

    # --- 6. ФИНАЛЬНАЯ СБОРКА И ВЫВОД ---
    # Таблица печатается как обычно, а summary_text подставляется уже в нужном формате (в рамке или без)
    final_output = f"{title}\n\n{markdown_table}\n{legend_text}{summary_text}"
    display(Markdown(final_output))

# Пример вызова
# Стандартный вызов (3 знака после запятой)
# print_regression_summary(res_hc)

# Округление до 4 знаков и уровень значимости 1%
# print_regression_summary(res_hc, sign_level=0.01, decimals=4)

# Вывод только таблицы с округлением до 2 знаков (без звездочек и текста)
# print_regression_summary(res_ols, decimals=2, show_signif=False, show_text=False)

In [12]:
#| echo: false
#| warning: false
#| output: asis

# Вывод результатов t-теста с уровнем значимости sign_level, звездочками, без текста и без рамки
print_regression_summary(res, sign_level=sign_level, show_signif=True, show_text=False, use_callout=False)

__Результаты $t$-теста для коэффициентов (неробастные стандартные ошибки, s.e.)__

|           |   Coef. |   Std.Err. |   t-value |   p-value | Signif.   |
|:----------|--------:|-----------:|----------:|----------:|:----------|
| Intercept |   0.864 |      0.022 |    39.444 |     0     |           |
| mortno    |   0.073 |      0.016 |     4.578 |     0     | ***       |
| unem      |  -0.006 |      0.003 |    -1.858 |     0.063 | *         |
| dep       |  -0.018 |      0.007 |    -2.57  |     0.01  | **        |
| male      |   0.002 |      0.02  |     0.094 |     0.925 |           |
| married   |   0.046 |      0.018 |     2.604 |     0.009 | ***       |
| yjob      |  -0.001 |      0.007 |    -0.099 |     0.921 |           |
| self      |  -0.036 |      0.022 |    -1.621 |     0.105 |           |

*Signif. codes:* `***` $p<0.01$, `**` $p<0.05$, `*` $p<0.1$

Какие коэффициенты значимы?

In [13]:
#| echo: false
#| warning: false
#| output: asis

from IPython.display import display, Markdown

def print_significant_coeffs(res, sign_level=0.05, use_callout=False, theme="note", title=None):
    """
    Выводит список статистически значимых коэффициентов регрессии (без константы).
    """
    # 1. Автоматически извлекаем тип стандартных ошибок из модели
    cov_type = getattr(res, 'cov_type', 'nonrobust')
    is_robust = cov_type != 'nonrobust'
    
    if is_robust:
        error_type = f"по робастным стандартным ошибкам {cov_type}"
    else:
        error_type = "по классическим стандартным ошибкам"

    # 2. Отбираем значимые коэффициенты
    all_sign_coeffs = res.params.index[res.pvalues < sign_level].tolist()

    # 3. Исключаем константу
    sign_coeffs = [coef for coef in all_sign_coeffs if coef not in ['const', 'Intercept']]

    # 4. Переводим уровень значимости в проценты
    percent_level = f"{sign_level * 100:g}"

    # --- 5. ФОРМИРУЕМ ЭЛЕМЕНТЫ ВЫВОДА ---
    
    # Базовый динамический заголовок
    dynamic_title = f"Коэффициенты, значимые на уровне {percent_level}% ({error_type})"
    
    # Заголовок, который реально будет использован (динамический, если не передан свой)
    actual_title = title if title is not None else dynamic_title

    # Содержимое (только переменные или короткое сообщение об отсутствии)
    if sign_coeffs:
        formatted_coeffs = ", ".join([f"`{coef}`" for coef in sign_coeffs])
        content_text = formatted_coeffs
    else:
        content_text = "Значимых коэффициентов не выявлено."

    # --- 6. СОБИРАЕМ РЕЗУЛЬТАТ С РАМКОЙ ИЛИ БЕЗ ---
    
    if use_callout:
        # В рамке: подробный текст уходит в заголовок, внутри только список переменных
        output_md = f"\n\n::: {{.callout-{theme} title=\"{actual_title}\" icon=\"true\"}}\n{content_text}\n:::\n\n"
    else:
        # Без рамки: все в одной строке (с жирным префиксом)
        output_md = f"\n\n**{actual_title}:**\n\n {content_text}\n\n"

    display(Markdown(output_md))

In [14]:
# Выводим значимые коэффициенты для модели с неробастными стандартными ошибками на уровне sign_level, в рамке
print_significant_coeffs(res, sign_level=sign_level, use_callout=True)



::: {.callout-note title="Коэффициенты, значимые на уровне 1% (по классическим стандартным ошибкам)" icon="true"}
`mortno`, `married`
:::



In [15]:
#| echo: false
#| warning: false
#| output: asis

# Вывод результатов t-теста с уровнем значимости sign_level, звездочками, без текста и без рамки
print_regression_summary(res_hc, sign_level=0.01, show_signif=True, show_text=False, use_callout=False)

__Результаты $t$-теста для коэффициентов (робастные стандартные ошибки, HC3-s.e.): __

|           |   Coef. |   Std.Err. |   t-value |   p-value | Signif.   |
|:----------|--------:|-----------:|----------:|----------:|:----------|
| Intercept |   0.864 |      0.023 |    37.135 |     0     |           |
| mortno    |   0.073 |      0.015 |     4.886 |     0     | ***       |
| unem      |  -0.006 |      0.004 |    -1.605 |     0.108 |           |
| dep       |  -0.018 |      0.008 |    -2.429 |     0.015 | **        |
| male      |   0.002 |      0.021 |     0.089 |     0.929 |           |
| married   |   0.046 |      0.019 |     2.458 |     0.014 | **        |
| yjob      |  -0.001 |      0.006 |    -0.107 |     0.915 |           |
| self      |  -0.036 |      0.025 |    -1.464 |     0.143 |           |

*Signif. codes:* `***` $p<0.01$, `**` $p<0.05$, `*` $p<0.1$

Какие коэффициенты значимы?

In [16]:
# Выводим значимые коэффициенты для модели с робастными стандартными ошибками HC3 на уровне sign_level, в рамке
print_significant_coeffs(res_hc, sign_level=sign_level, use_callout=True)



::: {.callout-note title="Коэффициенты, значимые на уровне 1% (по робастным стандартным ошибкам HC3)" icon="true"}
`mortno`
:::




::: {.callout-note}
## Расшифровка кодов статистической значимости (Signif. codes)

В таблицах регрессионного анализа звездочками традиционно обозначается $p$-значение (p-value) для каждого коэффициента. Чем меньше $p$-значение, тем сильнее статистические доказательства против нулевой гипотезы ($H_0$), согласно которой данный фактор не влияет на зависимую переменную (истинный коэффициент равен нулю).

* `***` : **$p < 0.01$** (от 0 до 0.01) — *высокая значимость*. Нулевая гипотеза отвергается на уровне значимости 1%.
* `**` : **$0.01 \le p < 0.05$** — *стандартная значимость*. Коэффициент значим на 5% уровне. Это самый распространенный порог для подтверждения гипотез в большинстве исследований.
* `*` : **$0.05 \le p < 0.1$** — *маргинальная (слабая) значимость*. Коэффициент значим лишь на 10% уровне. В строгих моделях предиктор признают незначимым, но иногда интерпретируют как «статистическую тенденцию».
* ` ` (пустое место) : **$p \ge 0.1$** (от 0.1 до 1) — *статистически не значимо*. У нас нет достаточных оснований утверждать, что переменная оказывает влияние на результат.
:::

## approve equation #2

Для датасета `loanapp` рассмотрим регрессию __approve на appinc, I(appinc ** 2), mortno, unem, dep, male, married, yjob, self__

Оцените модель на данных и укажите (робастные) коэффициенты подогнанной модели. __Ответ округлите до 3-х десятичных знаков.__

Ответ:

In [17]:
#| echo: false
#| warning: false
#| output: asis

res = ols(formula = 'approve ~ 1 + appinc + I(appinc ** 2) + mortno + unem + dep + male + married + yjob + self', data=loanapp).fit()
res_hc = ols(formula = 'approve ~ 1 + appinc + I(appinc ** 2) + mortno + unem + dep + male + married + yjob + self', data=loanapp).fit(cov_type='HC3')

digits = 3
# Формирование таблицы результатов
results_df = pd.DataFrame({
    'Переменная': res.params.index,
    #'Оценка (OLS)': np.round(res.params, digits),
    #'Стд. ошибка (OLS)': np.round(res.bse, digits),
    'Оценка (HC3)': np.round(res_hc.params, digits),
    #'Стд. ошибка (HC3)': np.round(res_hc.bse, digits)
})

print(results_df.to_markdown(index=False))

| Переменная     |   Оценка (HC3) |
|:---------------|---------------:|
| Intercept      |          0.842 |
| appinc         |          0.001 |
| I(appinc ** 2) |         -0     |
| mortno         |          0.066 |
| unem           |         -0.006 |
| dep            |         -0.017 |
| male           |         -0.003 |
| married        |          0.043 |
| yjob           |         -0.001 |
| self           |         -0.04  |


In [18]:
#| echo: false
#| warning: false
#| output: asis

# Уровень значимости
sign_level = 0.01

text = f"""
Модель была подогнана по {res.nobs:.0f} наблюдению. <span style="color: blue">Уровень значимости {sign_level * 100:.0f}%</span>
"""

display(Markdown(text))


Модель была подогнана по 1971 наблюдению. <span style="color: blue">Уровень значимости 1%</span>


Вычислите критическое значения для $t$-теста. __Ответ округлите до 3-х десятичных знаков.__

Ответ:

In [19]:
#| echo: false
#| warning: false
#| output: asis

# Уровень значимости
sign_level = 0.01
# Критическое значение t-распределения для двустороннего теста (делим уровень на 2) и соответствующих степеней свободы
t_crit = stats.t.isf(q=sign_level/2, df=res.df_resid)

# Формируем текст ответа с динамическим подставлением критического значения и уровня значимости
answer_text = f"$t_{{crit}} = {t_crit:.3f}$"

# Вывод критического значения в рамке с заголовком "Критическое значение"
display_answer(answer_text)



::: {.callout-note title="Критическое значение" icon="true"}
$t_{crit} = 2.578$
:::



__Укажите результаты робастного и неробастного $t$-теста. Ответ округлите до 3-х десятичных знаков.__

In [20]:
#| echo: false
#| warning: false
#| output: asis

# Вывод результатов t-теста с уровнем значимости sign_level, звездочками, без текста и без рамки
print_regression_summary(res, sign_level=sign_level, show_signif=True, show_text=False, use_callout=False)

__Результаты $t$-теста для коэффициентов (неробастные стандартные ошибки, s.e.)__

|                |   Coef. |   Std.Err. |   t-value |   p-value | Signif.   |
|:---------------|--------:|-----------:|----------:|----------:|:----------|
| Intercept      |   0.842 |      0.025 |    33.3   |     0     |           |
| appinc         |   0.001 |      0     |     2.082 |     0.037 | **        |
| I(appinc ** 2) |  -0     |      0     |    -2.803 |     0.005 | ***       |
| mortno         |   0.066 |      0.016 |     4.029 |     0     | ***       |
| unem           |  -0.006 |      0.003 |    -1.757 |     0.079 | *         |
| dep            |  -0.017 |      0.007 |    -2.374 |     0.018 | **        |
| male           |  -0.003 |      0.02  |    -0.142 |     0.887 |           |
| married        |   0.043 |      0.018 |     2.449 |     0.014 | **        |
| yjob           |  -0.001 |      0.007 |    -0.132 |     0.895 |           |
| self           |  -0.04  |      0.023 |    -1.786 |     0.074 | *         |

*Signif. codes:* `***` $p<0.01$, `**` $p<0.05$, `*` $p<0.1$

Какие коэффициенты значимы?

In [21]:
#| echo: false
#| warning: false
#| output: asis

# Выводим значимые коэффициенты для модели с неробастными стандартными ошибками на уровне sign_level, в рамке
print_significant_coeffs(res, sign_level=sign_level, use_callout=True)



::: {.callout-note title="Коэффициенты, значимые на уровне 1% (по классическим стандартным ошибкам)" icon="true"}
`I(appinc ** 2)`, `mortno`
:::



In [22]:
#| echo: false
#| warning: false
#| output: asis

# Вывод результатов t-теста с уровнем значимости sign_level, звездочками, без текста и без рамки
print_regression_summary(res_hc, sign_level=sign_level, show_signif=True, show_text=False, use_callout=False)

__Результаты $t$-теста для коэффициентов (робастные стандартные ошибки, HC3-s.e.): __

|                |   Coef. |   Std.Err. |   t-value |   p-value | Signif.   |
|:---------------|--------:|-----------:|----------:|----------:|:----------|
| Intercept      |   0.842 |      0.027 |    31.003 |     0     |           |
| appinc         |   0.001 |      0     |     1.958 |     0.05  | *         |
| I(appinc ** 2) |  -0     |      0     |    -2.374 |     0.018 | **        |
| mortno         |   0.066 |      0.015 |     4.321 |     0     | ***       |
| unem           |  -0.006 |      0.004 |    -1.515 |     0.13  |           |
| dep            |  -0.017 |      0.007 |    -2.28  |     0.023 | **        |
| male           |  -0.003 |      0.021 |    -0.135 |     0.893 |           |
| married        |   0.043 |      0.019 |     2.309 |     0.021 | **        |
| yjob           |  -0.001 |      0.006 |    -0.14  |     0.889 |           |
| self           |  -0.04  |      0.025 |    -1.602 |     0.109 |           |

*Signif. codes:* `***` $p<0.01$, `**` $p<0.05$, `*` $p<0.1$

Какие коэффициенты значимы?

In [23]:
#| echo: false
#| warning: false
#| output: asis

# Выводим значимые коэффициенты для модели с робастными стандартными ошибками на уровне sign_level, в рамке
print_significant_coeffs(res_hc, sign_level=sign_level, use_callout=True)



::: {.callout-note title="Коэффициенты, значимые на уровне 1% (по робастным стандартным ошибкам HC3)" icon="true"}
`mortno`
:::



## labour force equation #1

Для датасета `TableF5-1` рассмотрим регрессию __LFP на WA, I(WA ** 2), WE, KL6, K618, CIT, UN, np.log(FAMINC)__

Оцените модель на данных и укажите (робастные) коэффициенты подогнанной модели. __Ответ округлите до 3-х десятичных знаков.__

Ответ:

In [24]:
#| echo: false
#| warning: false
#| output: asis

sign_level = 0.01
res = ols(formula = 'LFP ~ 1 + WA + I(WA ** 2) + WE + KL6 + K618 + CIT + UN + np.log(FAMINC)', data=mroz_Greene).fit()
res_hc = ols(formula = 'LFP ~ 1 + WA + I(WA ** 2) + WE + KL6 + K618 + CIT + UN + np.log(FAMINC)', data=mroz_Greene).fit(cov_type='HC3')

digits = 3
# Формирование таблицы результатов
results_df = pd.DataFrame({
    'Переменная': res.params.index,
    #'Оценка (OLS)': np.round(res.params, digits),
    #'Стд. ошибка (OLS)': np.round(res.bse, digits),
    'Оценка (HC3)': np.round(res_hc.params, digits),
    #'Стд. ошибка (HC3)': np.round(res_hc.bse, digits)
})

print(results_df.to_markdown(index=False))

| Переменная     |   Оценка (HC3) |
|:---------------|---------------:|
| Intercept      |         -0.321 |
| WA             |          0.008 |
| I(WA ** 2)     |         -0     |
| WE             |          0.038 |
| KL6            |         -0.296 |
| K618           |         -0.021 |
| CIT            |         -0.048 |
| UN             |         -0.004 |
| np.log(FAMINC) |          0.072 |


In [25]:
#| echo: false
#| warning: false
#| output: asis

# Уровень значимости
sign_level = 0.01

text = f"""
Модель была подогнана по {res.nobs:.0f} наблюдению. <span style="color: blue">Уровень значимости {sign_level * 100:.0f}%</span>
"""

display(Markdown(text))


Модель была подогнана по 753 наблюдению. <span style="color: blue">Уровень значимости 1%</span>


Вычислите критическое значения для $t$-теста. __Ответ округлите до 3-х десятичных знаков.__

Ответ:

In [26]:
#| echo: false
#| warning: false
#| output: asis

# Уровень значимости
sign_level = 0.01
# Критическое значение t-распределения для двустороннего теста (делим уровень на 2) и соответствующих степеней свободы
t_crit = stats.t.isf(q=sign_level/2, df=res.df_resid)

# Формируем текст ответа с динамическим подставлением критического значения и уровня значимости
answer_text = f"$t_{{crit}} = {t_crit:.3f}$"

# Вывод критического значения в рамке с заголовком "Критическое значение"
display_answer(answer_text)



::: {.callout-note title="Критическое значение" icon="true"}
$t_{crit} = 2.582$
:::



__Укажите результаты робастного и неробастного $t$-теста. Ответ округлите до 3-х десятичных знаков.__

In [27]:
#| echo: false
#| warning: false
#| output: asis

# Вывод результатов t-теста с уровнем значимости sign_level, звездочками, без текста и без рамки
print_regression_summary(res, sign_level=sign_level, show_signif=True, show_text=False)

__Результаты $t$-теста для коэффициентов (неробастные стандартные ошибки, s.e.)__

|                |   Coef. |   Std.Err. |   t-value |   p-value | Signif.   |
|:---------------|--------:|-----------:|----------:|----------:|:----------|
| Intercept      |  -0.321 |      0.592 |    -0.542 |     0.588 |           |
| WA             |   0.008 |      0.025 |     0.305 |     0.76  |           |
| I(WA ** 2)     |  -0     |      0     |    -0.847 |     0.398 |           |
| WE             |   0.038 |      0.008 |     4.574 |     0     | ***       |
| KL6            |  -0.296 |      0.037 |    -8.011 |     0     | ***       |
| K618           |  -0.021 |      0.015 |    -1.434 |     0.152 |           |
| CIT            |  -0.048 |      0.038 |    -1.283 |     0.2   |           |
| UN             |  -0.004 |      0.006 |    -0.647 |     0.518 |           |
| np.log(FAMINC) |   0.072 |      0.037 |     1.965 |     0.05  | **        |

*Signif. codes:* `***` $p<0.01$, `**` $p<0.05$, `*` $p<0.1$

Какие коэффициенты значимы?

In [28]:
#| echo: false
#| warning: false
#| output: asis

# Выводим значимые коэффициенты для модели с неробастными стандартными ошибками на уровне sign_level, в рамке
print_significant_coeffs(res, sign_level=sign_level, use_callout=True)



::: {.callout-note title="Коэффициенты, значимые на уровне 1% (по классическим стандартным ошибкам)" icon="true"}
`WE`, `KL6`
:::



In [29]:
#| echo: false
#| warning: false
#| output: asis

# Вывод результатов t-теста с уровнем значимости sign_level, звездочками, без текста и без рамки
print_regression_summary(res_hc, sign_level=sign_level, show_signif=True, show_text=False)

__Результаты $t$-теста для коэффициентов (робастные стандартные ошибки, HC3-s.e.): __

|                |   Coef. |   Std.Err. |   t-value |   p-value | Signif.   |
|:---------------|--------:|-----------:|----------:|----------:|:----------|
| Intercept      |  -0.321 |      0.585 |    -0.549 |     0.583 |           |
| WA             |   0.008 |      0.024 |     0.307 |     0.759 |           |
| I(WA ** 2)     |  -0     |      0     |    -0.843 |     0.399 |           |
| WE             |   0.038 |      0.008 |     4.736 |     0     | ***       |
| KL6            |  -0.296 |      0.034 |    -8.763 |     0     | ***       |
| K618           |  -0.021 |      0.015 |    -1.409 |     0.159 |           |
| CIT            |  -0.048 |      0.037 |    -1.3   |     0.193 |           |
| UN             |  -0.004 |      0.006 |    -0.633 |     0.527 |           |
| np.log(FAMINC) |   0.072 |      0.038 |     1.921 |     0.055 | *         |

*Signif. codes:* `***` $p<0.01$, `**` $p<0.05$, `*` $p<0.1$

Какие коэффициенты значимы?

In [30]:
#| echo: false
#| warning: false
#| output: asis

# Выводим значимые коэффициенты для модели с робастными стандартными ошибками на уровне sign_level, в рамке
print_significant_coeffs(res_hc, sign_level=sign_level, use_callout=True)



::: {.callout-note title="Коэффициенты, значимые на уровне 1% (по робастным стандартным ошибкам HC3)" icon="true"}
`WE`, `KL6`
:::



## labour force equation #2

Для датасета `TableF5-1` рассмотрим регрессию __LFP на WA, WE, CIT, UN, np.log(FAMINC)__

Оцените модель на данных и укажите (робастные) коэффициенты подогнанной модели. __Ответ округлите до 3-х десятичных знаков.__

Ответ:

In [31]:
#| echo: false
#| warning: false
#| output: asis

sign_level = 0.05
res = ols(formula = 'LFP ~ 1 + WA + WE + CIT + UN + np.log(FAMINC)', data=mroz_Greene).fit()
res_hc = ols(formula = 'LFP ~ 1 + WA + WE + CIT + UN + np.log(FAMINC)', data=mroz_Greene).fit(cov_type='HC3')

digits = 3
# Формирование таблицы результатов
results_df = pd.DataFrame({
    'Переменная': res.params.index,
    #'Оценка (OLS)': np.round(res.params, digits),
    #'Стд. ошибка (OLS)': np.round(res.bse, digits),
    'Оценка (HC3)': np.round(res_hc.params, digits),
    #'Стд. ошибка (HC3)': np.round(res_hc.bse, digits)
})

print(results_df.to_markdown(index=False))

| Переменная     |   Оценка (HC3) |
|:---------------|---------------:|
| Intercept      |         -0.536 |
| WA             |         -0.004 |
| WE             |          0.033 |
| CIT            |         -0.048 |
| UN             |         -0.005 |
| np.log(FAMINC) |          0.094 |


In [32]:
#| echo: false
#| warning: false
#| output: asis

# Уровень значимости
sign_level = 0.05

text = f"""
Модель была подогнана по {res.nobs:.0f} наблюдению. <span style="color: blue">Уровень значимости {sign_level * 100:.0f}%</span>
"""

display(Markdown(text))


Модель была подогнана по 753 наблюдению. <span style="color: blue">Уровень значимости 5%</span>


Вычислите критическое значения для $t$-теста. __Ответ округлите до 3-х десятичных знаков.__

In [33]:
#| echo: false
#| warning: false
#| output: asis

# Уровень значимости
sign_level = 0.05
# Критическое значение t-распределения для двустороннего теста (делим уровень на 2) и соответствующих степеней свободы
t_crit = stats.t.isf(q=sign_level/2, df=res.df_resid)

# Формируем текст ответа с динамическим подставлением критического значения и уровня значимости
answer_text = f"$t_{{crit}} = {t_crit:.3f}$"

# Вывод критического значения в рамке с заголовком "Критическое значение"
display_answer(answer_text)



::: {.callout-note title="Критическое значение" icon="true"}
$t_{crit} = 1.963$
:::



__Укажите результаты робастного и неробастного $t$-теста. Ответ округлите до 3-х десятичных знаков.__

In [34]:
#| echo: false
#| warning: false
#| output: asis

# Вывод результатов t-теста с уровнем значимости sign_level, звездочками, без текста и без рамки
print_regression_summary(res, sign_level=sign_level, show_signif=True, show_text=False, use_callout=False)

__Результаты $t$-теста для коэффициентов (неробастные стандартные ошибки, s.e.)__

|                |   Coef. |   Std.Err. |   t-value |   p-value | Signif.   |
|:---------------|--------:|-----------:|----------:|----------:|:----------|
| Intercept      |  -0.536 |      0.36  |    -1.488 |     0.137 |           |
| WA             |  -0.004 |      0.002 |    -1.668 |     0.096 | *         |
| WE             |   0.033 |      0.009 |     3.914 |     0     | ***       |
| CIT            |  -0.048 |      0.039 |    -1.217 |     0.224 |           |
| UN             |  -0.005 |      0.006 |    -0.868 |     0.386 |           |
| np.log(FAMINC) |   0.094 |      0.038 |     2.449 |     0.015 | **        |

*Signif. codes:* `***` $p<0.01$, `**` $p<0.05$, `*` $p<0.1$

Какие коэффициенты значимы?

In [35]:
#| echo: false
#| warning: false
#| output: asis

# Выводим значимые коэффициенты для модели с неробастными стандартными ошибками на уровне sign_level, в рамке
print_significant_coeffs(res, sign_level=sign_level, use_callout=True)



::: {.callout-note title="Коэффициенты, значимые на уровне 5% (по классическим стандартным ошибкам)" icon="true"}
`WE`, `np.log(FAMINC)`
:::



In [36]:
#| echo: false
#| warning: false
#| output: asis

# Вывод результатов t-теста с уровнем значимости sign_level, звездочками, без текста и без рамки
print_regression_summary(res_hc, sign_level=sign_level, show_signif=True, show_text=False, use_callout=False)

__Результаты $t$-теста для коэффициентов (робастные стандартные ошибки, HC3-s.e.): __

|                |   Coef. |   Std.Err. |   t-value |   p-value | Signif.   |
|:---------------|--------:|-----------:|----------:|----------:|:----------|
| Intercept      |  -0.536 |      0.368 |    -1.457 |     0.145 |           |
| WA             |  -0.004 |      0.002 |    -1.635 |     0.102 |           |
| WE             |   0.033 |      0.008 |     4.036 |     0     | ***       |
| CIT            |  -0.048 |      0.039 |    -1.212 |     0.226 |           |
| UN             |  -0.005 |      0.006 |    -0.855 |     0.392 |           |
| np.log(FAMINC) |   0.094 |      0.04  |     2.351 |     0.019 | **        |

*Signif. codes:* `***` $p<0.01$, `**` $p<0.05$, `*` $p<0.1$

Какие коэффициенты значимы?

In [37]:
#| echo: false
#| warning: false
#| output: asis

# Выводим значимые коэффициенты для модели с робастными стандартными ошибками на уровне sign_level, в рамке
print_significant_coeffs(res_hc, sign_level=sign_level, use_callout=True)



::: {.callout-note title="Коэффициенты, значимые на уровне 5% (по робастным стандартным ошибкам HC3)" icon="true"}
`WE`, `np.log(FAMINC)`
:::



# F-тест: значимость регрессии

## approve equation #1

Для датасета `loanapp` рассмотрим регрессию __approve на unem, male, yjob, self__

Результаты оценивания:

In [38]:
#| echo: false
#| warning: false
#| output: asis

digits = 3

mod = ols(formula='approve ~ 1 + unem + male + yjob + self', data=loanapp)
res = mod.fit()
res_hc = mod.fit(cov_type='HC3')

# Собираем таблицу (зависимую переменную из info_dict убираем)
compare_table = summary_col(
    [res, res_hc], 
    model_names=['approve (OLS)', 'approve (HC3)'], # Это будет первая строка (заголовок таблицы)
    stars=True, 
    regressor_order=['unem', 'male', 'yjob', 'self', 'Intercept'],
    float_format=f'%0.{digits}f',
    # info_dict={
    #     'Nobs': lambda x: f"{int(x.nobs)}",
    #     'F-stat': lambda x: f"{x.fvalue:.{digits}f}",
    #     'Prob (F-stat)': lambda x: f"{x.f_pvalue:.{digits}f}",
    # }
)

# 1. Извлекаем готовую таблицу
df_compare = compare_table.tables[0]

# 2. Формируем безопасную легенду
legend_text = "\n\n*Signif. codes:* \\*\\*\\* $p<0.01$, \\*\\* $p<0.05$, \\* $p<0.1$"

# 3. Конвертируем DataFrame в Markdown и выводим
display(Markdown(df_compare.to_markdown() + legend_text))

|                | approve (OLS)   | approve (HC3)   |
|:---------------|:----------------|:----------------|
| unem           | -0.007**        | -0.007*         |
|                | (0.003)         | (0.004)         |
| male           | 0.021           | 0.021           |
|                | (0.019)         | (0.020)         |
| yjob           | 0.001           | 0.001           |
|                | (0.007)         | (0.006)         |
| self           | -0.030          | -0.030          |
|                | (0.022)         | (0.025)         |
| Intercept      | 0.891***        | 0.891***        |
|                | (0.021)         | (0.023)         |
| R-squared      | 0.004           | 0.004           |
| R-squared Adj. | 0.002           | 0.002           |

*Signif. codes:* \*\*\* $p<0.01$, \*\* $p<0.05$, \* $p<0.1$

<span style="color: blue">Зафиксируем уровень значимости 10%.</span>

Тестируется значимость регрессии, т.е. гипотеза $H_0:\beta_{unem}=\beta_{male}=\beta_{yjob}=\beta_{self}=0$. 

Вычислите критическое значение для гипотезы. __Ответ округлите до 3-х десятичных знаков.__

In [39]:
#| echo: false
#| warning: false
#| output: asis

digits = 3

# Задаем уровень значимости
sign_level = 0.10
# Критическое значение F-распределения
F_crit = f.isf(sign_level, mod.df_model, mod.df_resid)

# Формируем текст ответа с динамическим подставлением критического значения и уровня значимости
answer_text = f"$F_{{crit}} = {F_crit:.{digits}f}$"

# Вывод критического значения в рамке с заголовком "Критическое значение"
display_answer(answer_text)



::: {.callout-note title="Критическое значение" icon="true"}
$F_{crit} = 1.948$
:::



Получите результаты неробастного F-теста. __Ответ округлите до 3-х десятичных знаков.__

In [40]:
#| echo: false
#| warning: false
#| output: asis

digits = 3

# Формирование таблицы результатов
results_F = pd.DataFrame({
    'F-statistic': [round(res.fvalue, digits)],
    'Prob(F-statistic)': [round(res.f_pvalue, digits)]
})

print(results_F.to_markdown(index=False))

|   F-statistic |   Prob(F-statistic) |
|--------------:|--------------------:|
|         2.011 |                0.09 |


Какой можно сделать вывод?

In [41]:
#| echo: false
#| warning: false
#| output: asis

if res.f_pvalue < sign_level:
    text = "Регрессия **значима**."
    theme = "tip" # Зеленая рамка для успешного результата
else:
    text = "Регрессия **незначима**."
    theme = "important" # Красная рамка, если модель не работает

output_md = f"\n\n::: {{.callout-{theme} title=\"Вывод\" icon=\"true\"}}\n{text}\n:::\n\n"
display(Markdown(output_md))



::: {.callout-tip title="Вывод" icon="true"}
Регрессия **значима**.
:::



Получите результаты робастного F-теста. __Ответ округлите до 3-х десятичных знаков.__

In [42]:
#| echo: false
#| warning: false
#| output: asis

digits = 3

# Формирование таблицы результатов
results_F_hc = pd.DataFrame({
    'F-statistic': [round(res_hc.fvalue, digits)],
    'Prob(F-statistic)': [round(res_hc.f_pvalue, digits)]
})

print(results_F_hc.to_markdown(index=False))

|   F-statistic |   Prob(F-statistic) |
|--------------:|--------------------:|
|         1.667 |               0.155 |


Какой можно сделать вывод?

In [43]:
#| echo: false
#| warning: false
#| output: asis

if res_hc.f_pvalue < sign_level:
    text = "Регрессия **значима**."
    theme = "tip" # Зеленая рамка для успешного результата
else:
    text = "Регрессия **незначима**."
    theme = "important" # Красная рамка, если модель не работает

output_md = f"\n\n::: {{.callout-{theme} title=\"Вывод\" icon=\"true\"}}\n{text}\n:::\n\n"
display(Markdown(output_md))



::: {.callout-important title="Вывод" icon="true"}
Регрессия **незначима**.
:::




## approve equation #2

Для датасета `loanapp` рассмотрим регрессию __approve на appinc, I(appinc ** 2), mortno, dep__

In [44]:
#| echo: false
#| warning: false
#| output: asis

digits = 3

mod = ols(formula='approve ~ 1 + appinc + I(appinc ** 2) + mortno + unem', data=loanapp)
res = mod.fit()
res_hc = mod.fit(cov_type='HC3')

# Собираем их в одну красивую таблицу
compare_table = summary_col(
    [res, res_hc], 
    # Пишем зависимую переменную прямо в заголовке
    model_names=['OLS', 'HC3'],
    #model_names=['approve (OLS)', 'approve (HC3)'],
    stars=True, 
    regressor_order=['unem', 'male', 'yjob', 'self', 'Intercept'],
    float_format=f'%0.{digits}f', # Округляет коэффициенты и ст. ошибки
    #info_dict={
    #     # Вытаскиваем имя зависимой переменной:
    #    'Dep. Variable': lambda x: x.model.endog_names,
    #     'F-stat': lambda x: f"{x.fvalue:.{digits}f}",
    #     'Prob (F-stat)': lambda x: f"{x.f_pvalue:.{digits}f}",
    #     'Nobs': lambda x: f"{int(x.nobs)}",
    #}
)

# 1. Извлекаем готовую таблицу (pandas DataFrame) из объекта Summary
df_compare = compare_table.tables[0]

# 2. Формируем нашу безопасную для XeLaTeX легенду со звездочками
legend_text = "\n\n*Signif. codes:* \\*\\*\\* $p<0.01$, \\*\\* $p<0.05$, \\* $p<0.1$"

# 3. Конвертируем DataFrame в Markdown и выводим с легендой
display(Markdown(df_compare.to_markdown() + legend_text))

|                | OLS       | HC3      |
|:---------------|:----------|:---------|
| unem           | -0.007**  | -0.007*  |
|                | (0.003)   | (0.004)  |
| Intercept      | 0.856***  | 0.856*** |
|                | (0.022)   | (0.023)  |
| appinc         | 0.000**   | 0.000**  |
|                | (0.000)   | (0.000)  |
| I(appinc ** 2) | -0.000*** | -0.000** |
|                | (0.000)   | (0.000)  |
| mortno         | 0.066***  | 0.066*** |
|                | (0.016)   | (0.015)  |
| R-squared      | 0.019     | 0.019    |
| R-squared Adj. | 0.017     | 0.017    |

*Signif. codes:* \*\*\* $p<0.01$, \*\* $p<0.05$, \* $p<0.1$

<span style="color: blue">Зафиксируем уровень значимости 5%.</span>

Тестируется значимость регрессии, т.е. гипотеза $H_0:\beta_{appinc}=\beta_{appinc^2}=\beta_{mortno}=\beta_{dep}=0$.

Вычислите критическое значение для гипотезы. __Ответ округлите до 3-х десятичных знаков.__

In [45]:
#| echo: false
#| warning: false
#| output: asis

digits = 3

# Задаем уровень значимости
sign_level = 0.05
# Критическое значение F-распределения
F_crit = f.isf(sign_level, mod.df_model, mod.df_resid)

# Формируем текст ответа с динамическим подставлением критического значения и уровня значимости
answer_text = f"$F_{{crit}} = {F_crit:.{digits}f}$"

# Вывод критического значения в рамке с заголовком "Критическое значение"
display_answer(answer_text)



::: {.callout-note title="Критическое значение" icon="true"}
$F_{crit} = 2.376$
:::



Получите результаты неробастного F-теста. __Ответ округлите до 3-х десятичных знаков.__

In [46]:
#| echo: false
#| warning: false
#| output: asis

digits = 3

# Формирование таблицы результатов
results_F = pd.DataFrame({
    'F-statistic': [round(res.fvalue, digits)],
    'Prob(F-statistic)': [round(res.f_pvalue, digits)]
})

print(results_F.to_markdown(index=False))

|   F-statistic |   Prob(F-statistic) |
|--------------:|--------------------:|
|         9.606 |                   0 |


Какой можно сделать вывод?

In [47]:
#| echo: false
#| warning: false
#| output: asis

if res.f_pvalue < sign_level:
    text = "Регрессия **значима**."
    theme = "tip" # Зеленая рамка для успешного результата
else:
    text = "Регрессия **незначима**."
    theme = "important" # Красная рамка, если модель не работает

output_md = f"\n\n::: {{.callout-{theme} title=\"Вывод\" icon=\"true\"}}\n{text}\n:::\n\n"
display(Markdown(output_md))



::: {.callout-tip title="Вывод" icon="true"}
Регрессия **значима**.
:::



Получите результаты робастного F-теста. __Ответ округлите до 3-х десятичных знаков.__

In [48]:
#| echo: false
#| warning: false
#| output: asis

digits = 3

# Формирование таблицы результатов
results_F_hc = pd.DataFrame({
    'F-statistic': [round(res_hc.fvalue, digits)],
    'Prob(F-statistic)': [round(res_hc.f_pvalue, digits)]
})

print(results_F_hc.to_markdown(index=False))

|   F-statistic |   Prob(F-statistic) |
|--------------:|--------------------:|
|          9.13 |                   0 |


Какой можно сделать вывод?

In [49]:
#| echo: false
#| warning: false
#| output: asis

if res_hc.f_pvalue < sign_level:
    text = "Регрессия **значима**."
    theme = "tip" # Зеленая рамка для успешного результата
else:
    text = "Регрессия **незначима**."
    theme = "important" # Красная рамка, если модель не работает

output_md = f"\n\n::: {{.callout-{theme} title=\"Вывод\" icon=\"true\"}}\n{text}\n:::\n\n"
display(Markdown(output_md))



::: {.callout-tip title="Вывод" icon="true"}
Регрессия **значима**.
:::



## approve equation #3

Для датасета `loanapp` рассмотрим регрессию __approve на dep, male, married__

In [50]:
#| echo: false
#| warning: false
#| output: asis

digits = 3

mod = ols(formula='approve ~ 1 + dep + male + married', data=loanapp)
res = mod.fit()
res_hc = mod.fit(cov_type='HC3')

# Собираем их в одну красивую таблицу
compare_table = summary_col(
    [res, res_hc], 
    # Пишем зависимую переменную прямо в заголовке
    model_names=['OLS', 'HC3'],
    #model_names=['approve (OLS)', 'approve (HC3)'],
    stars=True, 
    regressor_order=['dep', 'male', 'married', 'Intercept'],
    float_format=f'%0.{digits}f', # Округляет коэффициенты и ст. ошибки
)

# 1. Извлекаем готовую таблицу (pandas DataFrame) из объекта Summary
df_compare = compare_table.tables[0]

# 2. Формируем нашу безопасную для XeLaTeX легенду со звездочками
legend_text = "\n\n*Signif. codes:* \\*\\*\\* $p<0.01$, \\*\\* $p<0.05$, \\* $p<0.1$"

# 3. Конвертируем DataFrame в Markdown и выводим с легендой
display(Markdown(df_compare.to_markdown() + legend_text))

|                | OLS      | HC3      |
|:---------------|:---------|:---------|
| dep            | -0.016** | -0.016** |
|                | (0.007)  | (0.008)  |
| male           | -0.002   | -0.002   |
|                | (0.020)  | (0.021)  |
| married        | 0.057*** | 0.057*** |
|                | (0.018)  | (0.019)  |
| Intercept      | 0.852*** | 0.852*** |
|                | (0.018)  | (0.019)  |
| R-squared      | 0.006    | 0.006    |
| R-squared Adj. | 0.005    | 0.005    |

*Signif. codes:* \*\*\* $p<0.01$, \*\* $p<0.05$, \* $p<0.1$

<span style="color: blue">Зафиксируем уровень значимости 1%.</span>

Тестируется значимость регрессии, т.е. гипотеза $H_0:\beta_{dep}=\beta_{male}=\beta_{married}=0$.

Вычислите критическое значение для гипотезы. __Ответ округлите до 3-х десятичных знаков.__

In [51]:
#| echo: false
#| warning: false
#| output: asis

digits = 3

# Задаем уровень значимости
sign_level = 0.01
# Критическое значение F-распределения
F_crit = f.isf(sign_level, res.df_model, res.df_resid)

# Формируем текст ответа с динамическим подставлением критического значения и уровня значимости
answer_text = f"$F_{{crit}} = {F_crit:.{digits}f}$"

# Вывод критического значения в рамке с заголовком "Критическое значение"
display_answer(answer_text)



::: {.callout-note title="Критическое значение" icon="true"}
$F_{crit} = 3.792$
:::



Получите результаты неробастного F-теста. __Ответ округлите до 3-х десятичных знаков.__

In [52]:
#| echo: false
#| warning: false
#| output: asis

digits = 3

# Формирование таблицы результатов
results_F = pd.DataFrame({
    'F-statistic': [round(res.fvalue, digits)],
    'Prob(F-statistic)': [round(res.f_pvalue, digits)]
})

print(results_F.to_markdown(index=False))

|   F-statistic |   Prob(F-statistic) |
|--------------:|--------------------:|
|         4.155 |               0.006 |


Какой можно сделать вывод?

In [53]:
#| echo: false
#| warning: false
#| output: asis

if res.f_pvalue < sign_level:
    text = "Регрессия **значима**."
    theme = "tip" # Зеленая рамка для успешного результата
else:
    text = "Регрессия **незначима**."
    theme = "important" # Красная рамка, если модель не работает

output_md = f"\n\n::: {{.callout-{theme} title=\"Вывод\" icon=\"true\"}}\n{text}\n:::\n\n"
display(Markdown(output_md))



::: {.callout-tip title="Вывод" icon="true"}
Регрессия **значима**.
:::



Получите результаты робастного F-теста. __Ответ округлите до 3-х десятичных знаков.__

In [54]:
#| echo: false
#| warning: false
#| output: asis

digits = 3

# Формирование таблицы результатов
results_F_hc = pd.DataFrame({
    'F-statistic': [round(res_hc.fvalue, digits)],
    'Prob(F-statistic)': [round(res_hc.f_pvalue, digits)]
})

print(results_F_hc.to_markdown(index=False))

|   F-statistic |   Prob(F-statistic) |
|--------------:|--------------------:|
|         3.724 |               0.011 |


Какой можно сделать вывод?

In [55]:
#| echo: false
#| warning: false
#| output: asis

if res_hc.f_pvalue < sign_level:
    text = "Регрессия **значима**."
    theme = "tip" # Зеленая рамка для успешного результата
else:
    text = "Регрессия **незначима**."
    theme = "important" # Красная рамка, если модель не работает

output_md = f"\n\n::: {{.callout-{theme} title=\"Вывод\" icon=\"true\"}}\n{text}\n:::\n\n"
display(Markdown(output_md))



::: {.callout-important title="Вывод" icon="true"}
Регрессия **незначима**.
:::



# F-тест: совместная значимость

## approve equation #1

Для датасета `loanapp` рассмотрим регрессию __approve на appinc, I(appinc ** 2), mortno, unem, dep, male, married, yjob, self__

In [56]:
#| echo: false
#| warning: false
#| output: asis

digits = 3



mod = ols(formula='approve ~ 1 + appinc + I(appinc ** 2) + mortno  + unem + dep + male + married + yjob + self', data=loanapp)
res = mod.fit()
res_hc = mod.fit(cov_type='HC3')

restricted_1 = ols(formula='approve ~ 1 + mortno  + unem + dep + male + married + yjob + self', data=loanapp).fit(cov_type='HC3')
restricted_2 = ols(formula='approve ~ 1 + appinc + I(appinc ** 2) + mortno + male + yjob + self', data=loanapp).fit(cov_type='HC3')


# 1. Упаковываем модели и их будущие названия в списки
models_list = [res_hc, restricted_1, restricted_2]
model_titles = ['Model 1', 'Model 2', 'Model 3']

# 2. Собираем базовую таблицу
compare_table = summary_col(
    models_list, 
    model_names=model_titles,
    stars=True, 
    regressor_order=['appinc', 'I(appinc ** 2)', 'mortno', 'unem', 'dep', 'male', 'married', 'yjob', 'self', 'Intercept'],
    float_format=f'%0.{digits}f',
    info_dict={
        'Nobs': lambda x: f"{int(x.nobs)}",
        'F-statistic': lambda x: f"{x.fvalue:.{digits}f}"
    }
)

df_compare = compare_table.tables[0]

# 3. Собираем словарь, где ключ — имя колонки, а значение — зависимая переменная нужной модели
# zip объединяет колонки таблицы и список моделей попарно
dep_vars_dict = {
    col_name: [model.model.endog_names] 
    for col_name, model in zip(df_compare.columns, models_list)
}

# Создаем верхнюю строку из этого словаря
top_row = pd.DataFrame(dep_vars_dict, index=['Dep. Variable'])

# 4. Приклеиваем строку наверх
df_compare = pd.concat([top_row, df_compare])

# 5. Выводим результат
legend_text = "\n\n*Signif. codes:* \\*\\*\\* $p<0.01$, \\*\\* $p<0.05$, \\* $p<0.1$"
display(Markdown(df_compare.to_markdown() + legend_text))

|                | Model 1   | Model 2   | Model 3   |
|:---------------|:----------|:----------|:----------|
| Dep. Variable  | approve   | approve   | approve   |
| appinc         | 0.001*    |           | 0.001**   |
|                | (0.000)   |           | (0.000)   |
| I(appinc ** 2) | -0.000**  |           | -0.000*** |
|                | (0.000)   |           | (0.000)   |
| mortno         | 0.066***  | 0.073***  | 0.067***  |
|                | (0.015)   | (0.015)   | (0.015)   |
| unem           | -0.006    | -0.006    |           |
|                | (0.004)   | (0.004)   |           |
| dep            | -0.017**  | -0.018**  |           |
|                | (0.007)   | (0.008)   |           |
| male           | -0.003    | 0.002     | 0.003     |
|                | (0.021)   | (0.021)   | (0.020)   |
| married        | 0.043**   | 0.046**   |           |
|                | (0.019)   | (0.019)   |           |
| yjob           | -0.001    | -0.001    | -0.000    |
|                | (0.006)   | (0.006)   | (0.006)   |
| self           | -0.040    | -0.036    | -0.050**  |
|                | (0.025)   | (0.025)   | (0.024)   |
| Intercept      | 0.842***  | 0.864***  | 0.824***  |
|                | (0.027)   | (0.023)   | (0.023)   |
| R-squared      | 0.025     | 0.020     | 0.019     |
| R-squared Adj. | 0.021     | 0.017     | 0.016     |
| F-statistic    | 5.032     | 5.849     | 6.172     |
| Nobs           | 1971      | 1971      | 1971      |

*Signif. codes:* \*\*\* $p<0.01$, \*\* $p<0.05$, \* $p<0.1$

### Гипотеза 1: значимость влияния дохода

Тестируется значимость влияния дохода, т.е. гипотеза $H_0:\beta_{appinc}=\beta_{appinc^2}=0$. 

<span style="color: blue">Зафиксируем уровень значимости 5%.</span>

Вычислите критическое значение для гипотезы. __Ответ округлите до 3-х десятичных знаков.__

In [57]:
#| echo: false
#| warning: false
#| output: asis

digits = 3

# Задаем уровень значимости
sign_level = 0.05
# Критическое значение F-распределения
F_crit = f.isf(sign_level, dfn = 2, dfd = mod.df_resid)

# Формируем текст ответа с динамическим подставлением критического значения и уровня значимости
answer_text = f"$F_{{crit}} = {F_crit:.{digits}f}$"

# Вывод критического значения в рамке с заголовком "Критическое значение"
display_answer(answer_text)



::: {.callout-note title="Критическое значение" icon="true"}
$F_{crit} = 3.000$
:::



Получите результаты неробастного F-теста. __Ответ округлите до 3-х десятичных знаков.__

In [58]:
#| echo: false
#| warning: false
#| output: asis

digits = 3

F_test = res.wald_test('appinc=0, I(appinc ** 2)=0', use_f=True)
# Формирование таблицы результатов
results_F = pd.DataFrame({
    'F-statistic': [np.round(F_test.statistic, digits)[0][0]],
    'Prob(F-statistic)': [np.round(F_test.pvalue, digits)]
})

print(results_F.to_markdown(index=False))

|   F-statistic |   Prob(F-statistic) |
|--------------:|--------------------:|
|         5.149 |               0.006 |


Какой можно сделать вывод?

In [59]:
#| echo: false
#| warning: false
#| output: asis

if F_test.pvalue < sign_level:
    text = "Гипотеза **отвергается**."
    theme = "tip" # Зеленая рамка для успешного результата
else:
    text = "Гипотеза **не отвергается**."
    theme = "important" # Красная рамка, если модель не работает

output_md = f"\n\n::: {{.callout-{theme} title=\"Вывод\" icon=\"true\"}}\n{text}\n:::\n\n"
display(Markdown(output_md))



::: {.callout-tip title="Вывод" icon="true"}
Гипотеза **отвергается**.
:::



Получите результаты робастного F-теста. __Ответ округлите до 3-х десятичных знаков.__

In [60]:
#| echo: false
#| warning: false
#| output: asis

digits = 3

F_test_hc = res_hc.wald_test('appinc=0, I(appinc ** 2)=0', use_f=True)
# Формирование таблицы результатов
results_F_hc = pd.DataFrame({
    'F-statistic': [np.round(F_test_hc.statistic, digits)[0][0]],
    'Prob(F-statistic)': [np.round(F_test_hc.pvalue, digits)]
})

print(results_F_hc.to_markdown(index=False))

|   F-statistic |   Prob(F-statistic) |
|--------------:|--------------------:|
|         2.961 |               0.052 |


Какой можно сделать вывод?

In [61]:
#| echo: false
#| warning: false
#| output: asis

if F_test_hc.pvalue < sign_level:
    text = "Гипотеза **отвергается**."
    theme = "tip" # Зеленая рамка для успешного результата
else:
    text = "Гипотеза **не отвергается**."
    theme = "important" # Красная рамка, если модель не работает

output_md = f"\n\n::: {{.callout-{theme} title=\"Вывод\" icon=\"true\"}}\n{text}\n:::\n\n"
display(Markdown(output_md))



::: {.callout-important title="Вывод" icon="true"}
Гипотеза **не отвергается**.
:::



### Гипотеза 2: совместная значимость unem, dep, married

Тестируется совместная значимость `unem`, `dep` и `married`, т.е. гипотеза $H_0:\beta_{unem}=\beta_{dep}=\beta_{married}=0$. 

<span style="color: blue">Зафиксируем уровень значимости 1%.</span>

Вычислите критическое значение для гипотезы. __Ответ округлите до 3-х десятичных знаков.__

In [62]:
#| echo: false
#| warning: false
#| output: asis

digits = 3

# Задаем уровень значимости
sign_level = 0.01
# Критическое значение F-распределения
F_crit = f.isf(sign_level, dfn = 3, dfd = mod.df_resid)

# Формируем текст ответа с динамическим подставлением критического значения и уровня значимости
answer_text = f"$F_{{crit}} = {F_crit:.{digits}f}$"

# Вывод критического значения в рамке с заголовком "Критическое значение"
display_answer(answer_text)



::: {.callout-note title="Критическое значение" icon="true"}
$F_{crit} = 3.792$
:::



Получите результаты неробастного F-теста. __Ответ округлите до 3-х десятичных знаков.__

In [63]:
#| echo: false
#| warning: false
#| output: asis

digits = 3

F_test = res.wald_test('unem=0, dep=0, married=0', use_f=True)
# Формирование таблицы результатов
results_F = pd.DataFrame({
    'F-statistic': [np.round(F_test.statistic, digits)[0][0]],
    'Prob(F-statistic)': [np.round(F_test.pvalue, digits)]
})

print(results_F.to_markdown(index=False))

|   F-statistic |   Prob(F-statistic) |
|--------------:|--------------------:|
|         4.054 |               0.007 |


Какой можно сделать вывод?

In [64]:
#| echo: false
#| warning: false
#| output: asis

if F_test.pvalue < sign_level:
    text = "Гипотеза **отвергается** — переменные **значимы совместно**."
    theme = "tip" # Зеленая рамка для успешного результата
else:
    text = "Гипотеза **не отвергается** — переменные **незначимы совместно**."
    theme = "important" # Красная рамка, если модель не работает

output_md = f"\n\n::: {{.callout-{theme} title=\"Вывод\" icon=\"true\"}}\n{text}\n:::\n\n"
display(Markdown(output_md))



::: {.callout-tip title="Вывод" icon="true"}
Гипотеза **отвергается** — переменные **значимы совместно**.
:::



Получите результаты робастного F-теста. __Ответ округлите до 3-х десятичных знаков.__

In [65]:
#| echo: false
#| warning: false
#| output: asis

digits = 3

F_test_hc = res_hc.wald_test('unem=0, dep=0, married=0', use_f=True)
# Формирование таблицы результатов
results_F_hc = pd.DataFrame({
    'F-statistic': [np.round(F_test_hc.statistic, digits)[0][0]],
    'Prob(F-statistic)': [np.round(F_test_hc.pvalue, digits)]
})

print(results_F_hc.to_markdown(index=False))

|   F-statistic |   Prob(F-statistic) |
|--------------:|--------------------:|
|         3.469 |               0.016 |


Какой можно сделать вывод?

In [66]:
#| echo: false
#| warning: false
#| output: asis

if F_test_hc.pvalue < sign_level:
    text = "Гипотеза **отвергается** — переменные **значимы совместно**."
    theme = "tip" # Зеленая рамка для успешного результата
else:
    text = "Гипотеза **не отвергается** — переменные **незначимы совместно**."
    theme = "important" # Красная рамка, если модель не работает

output_md = f"\n\n::: {{.callout-{theme} title=\"Вывод\" icon=\"true\"}}\n{text}\n:::\n\n"
display(Markdown(output_md))



::: {.callout-important title="Вывод" icon="true"}
Гипотеза **не отвергается** — переменные **незначимы совместно**.
:::



# Прогноз

## approve equation

Для датасета `loanapp` рассмотрим регрессию __approve на mortno, unem, dep, married__

In [67]:
#| echo: false
#| warning: false
#| output: asis

digits = 3

mod = ols(formula='approve ~ 1 + mortno + unem + dep + married', data=loanapp)
res_hc = mod.fit(cov_type='HC3')

# Собираем их в одну красивую таблицу
compare_table = summary_col(
    [res_hc], 
    # Пишем зависимую переменную прямо в заголовке
    model_names=['approve (HC3)'],
    stars=True, 
    regressor_order=['mortno', 'unem', 'dep', 'married', 'Intercept'],
    float_format=f'%0.{digits}f', # Округляет коэффициенты и ст. ошибки
    info_dict={
        'Nobs': lambda x: f"{int(x.nobs)}",
        'F-stat': lambda x: f"{x.fvalue:.{digits}f}",
        'Prob(F-stat)': lambda x: f"{x.f_pvalue:.{digits}f}",
    }
)

# 1. Извлекаем готовую таблицу (pandas DataFrame) из объекта Summary
df_compare = compare_table.tables[0]

# 2. Формируем нашу безопасную для XeLaTeX легенду со звездочками
legend_text = "\n\n*Signif. codes:* \\*\\*\\* $p<0.01$, \\*\\* $p<0.05$, \\* $p<0.1$"

# 3. Конвертируем DataFrame в Markdown и выводим с легендой
display(Markdown(df_compare.to_markdown() + legend_text))

|                | approve (HC3)   |
|:---------------|:----------------|
| mortno         | 0.071***        |
|                | (0.015)         |
| unem           | -0.007*         |
|                | (0.004)         |
| dep            | -0.019**        |
|                | (0.008)         |
| married        | 0.046***        |
|                | (0.018)         |
| Intercept      | 0.865***        |
|                | (0.020)         |
| R-squared      | 0.019           |
| R-squared Adj. | 0.017           |
| Nobs           | 1971            |
| F-stat         | 9.475           |
| Prob(F-stat)   | 0.000           |

*Signif. codes:* \*\*\* $p<0.01$, \*\* $p<0.05$, \* $p<0.1$

Рассмотрим трех людей с характеристиками

In [68]:
#| echo: false
#| warning: false
#| output: asis

# Данные для прогнозирования
new_df = pd.DataFrame({
    'mortno': [1, 1, 0],
    'unem': [3.2, 3.9, 1.8], 
    'dep': [0, 1, 0],
    'married': [1, 0, 1]
})

#print(new_df.to_markdown(index=False))
# Сдвигаем индекс на +1
new_df.index = new_df.index + 1
# 
new_df.index.name = 'Человек'

# Выводим в Quarto
display(Markdown(new_df.to_markdown(index=True)))

|   Человек |   mortno |   unem |   dep |   married |
|----------:|---------:|-------:|------:|----------:|
|         1 |        1 |    3.2 |     0 |         1 |
|         2 |        1 |    3.9 |     1 |         0 |
|         3 |        0 |    1.8 |     0 |         1 |

Вычислите прогноз для каждого человека по подогнанной модели. __Ответ округлите до 3-х десятичных знаков.__

In [69]:
#| echo: false
#| warning: false
#| output: asis

digits = 3

# Прогнозы
predictions = res_hc.predict(new_df)

# Собираем результаты в DataFrame
pred_df = pd.DataFrame({
    'Человек': [f"{i+1}" for i in range(len(predictions))],
    'Прогноз': [f"{pred:.{digits}f}" for pred in predictions]
})

# Формируем итоговый текст с заголовком и таблицей
output_md = "**Прогнозы вероятности одобрения кредита ${\\textrm P}(approve=1)$:**\n\n" + pred_df.to_markdown(index=False)

# Выводим в Quarto
display(Markdown(output_md))

**Прогнозы вероятности одобрения кредита ${\textrm P}(approve=1)$:**

|   Человек |   Прогноз |
|----------:|----------:|
|         1 |     0.959 |
|         2 |     0.889 |
|         3 |     0.898 |

## labour force equation

Для датасета `TableF5-1` рассмотрим регрессию __LFP на WA, I(WA ** 2), CIT, UN, np.log(FAMINC)__

In [70]:
#| echo: false
#| warning: false
#| output: asis

digits = 3

mod = ols(formula='LFP ~ 1 + WA + I(WA ** 2) + CIT + UN + np.log(FAMINC)', data=mroz_Greene)
res_hc = mod.fit(cov_type='HC3')

# Собираем их в одну красивую таблицу
compare_table = summary_col(
    [res_hc], 
    # Пишем зависимую переменную прямо в заголовке
    model_names=['LFP (HC3)'],
    stars=True, 
    regressor_order=['WA', 'I(WA ** 2)', 'CIT', 'UN', 'np.log(FAMINC)', 'Intercept'],
    float_format=f'%0.{digits}f', # Округляет коэффициенты и ст. ошибки
    info_dict={
        'Nobs': lambda x: f"{int(x.nobs)}",
        'F-stat': lambda x: f"{x.fvalue:.{digits}f}",
        'Prob(F-stat)': lambda x: f"{x.f_pvalue:.{digits}f}",
    }
)

# 1. Извлекаем готовую таблицу (pandas DataFrame) из объекта Summary
df_compare = compare_table.tables[0]

# 2. Формируем нашу безопасную для XeLaTeX легенду со звездочками
legend_text = "\n\n*Signif. codes:* \\*\\*\\* $p<0.01$, \\*\\* $p<0.05$, \\* $p<0.1$"

# 3. Конвертируем DataFrame в Markdown и выводим с легендой
display(Markdown(df_compare.to_markdown() + legend_text))

|                | LFP (HC3)   |
|:---------------|:------------|
| WA             | 0.042*      |
|                | (0.025)     |
| I(WA ** 2)     | -0.001*     |
|                | (0.000)     |
| CIT            | -0.039      |
|                | (0.040)     |
| UN             | -0.004      |
|                | (0.006)     |
| np.log(FAMINC) | 0.140***    |
|                | (0.037)     |
| Intercept      | -1.525**    |
|                | (0.600)     |
| R-squared      | 0.033       |
| R-squared Adj. | 0.027       |
| Nobs           | 753         |
| F-stat         | 4.994       |
| Prob(F-stat)   | 0.000       |

*Signif. codes:* \*\*\* $p<0.01$, \*\* $p<0.05$, \* $p<0.1$

Рассмотрим трех людей с характеристиками

In [71]:
#| echo: false
#| warning: false
#| output: asis

# Данные для прогнозирования
new_df = pd.DataFrame({
    'WA': [34, 36, 42], 
    'CIT': [1, 0, 0], 
    'UN': [3, 5, 7.5], 
    'FAMINC': [35000, 48500, 67800]
})

#print(new_df.to_markdown(index=False))
# Сдвигаем индекс на +1
new_df.index = new_df.index + 1
# 
new_df.index.name = 'Человек'

# Выводим в Quarto
display(Markdown(new_df.to_markdown(index=True)))

|   Человек |   WA |   CIT |   UN |   FAMINC |
|----------:|-----:|------:|-----:|---------:|
|         1 |   34 |     1 |  3   |    35000 |
|         2 |   36 |     0 |  5   |    48500 |
|         3 |   42 |     0 |  7.5 |    67800 |

Вычислите прогноз для каждого человека по подогнанной модели. __Ответ округлите до 3-х десятичных знаков.__

In [72]:
#| echo: false
#| warning: false
#| output: asis

digits = 3

# Прогнозы
predictions = res_hc.predict(new_df)

# Собираем результаты в DataFrame
pred_df = pd.DataFrame({
    'Человек': [f"{i+1}" for i in range(len(predictions))],
    'Прогноз': [f"{pred:.{digits}f}" for pred in predictions]
})

# Формируем итоговый текст с заголовком и таблицей
output_md = "**Прогнозы вероятности участия в рабочей силе ${\\textrm P}(LFP=1)$:**\n\n" + pred_df.to_markdown(index=False)

# Выводим в Quarto
display(Markdown(output_md))

**Прогнозы вероятности участия в рабочей силе ${\textrm P}(LFP=1)$:**

|   Человек |   Прогноз |
|----------:|----------:|
|         1 |     0.686 |
|         2 |     0.771 |
|         3 |     0.804 |

# Вопросы адекватности модели

## approve equation

Для датасета `loanapp` рассмотрим регрессию __approve на appinc, I(appinc ** 2), mortno, unem, dep, male, married, yjob, self__

In [73]:
#| echo: false
#| warning: false
#| message: false
#| output: asis

# 1. Строим модель с робастными ошибками HC3
res_hc = ols(formula='approve ~ 1 + appinc + I(appinc ** 2) + mortno + unem + dep + male + married + yjob + self', data=loanapp).fit(cov_type='HC3')

# 2. Извлекаем фактические значения, прогнозы и матрицу X прямо из модели
actual = res_hc.model.endog
fitted = res_hc.fittedvalues
df_model = res_hc.model.data.frame.copy()

# 3. Считаем предсказательную силу (Accuracy) по порогу 0.5
threshold = 0.5
predicted_classes = (fitted >= threshold).astype(int)
accuracy = (predicted_classes == actual).mean()

# Выводим точность красивым текстом
display(Markdown(f"**Доля правильных прогнозов (Accuracy) на обучающей выборке:** `{accuracy:.1%}`\n"))

# ВАЖНО: теперь мы ссылаемся на нашу новую колонку 'log_FAMINC'
x_vars = ['appinc', 'dep', 'yjob']

# 4. Строим графики в цикле

for i, var in enumerate(x_vars):
    # Текстовое пояснение перед картинкой (по желанию можно убрать, если хватает title)
    display(Markdown(f"<br>\n\n### Анализ переменной `{x_vars[i]}`"))
    
    # Создаем новую отдельную фигуру
    plt.figure(figsize=(8, 4))
    
    # Рисуем
    plt.scatter(df_model[var], actual, color='blue', s=15, alpha=0.4, label='Фактические (approve)')
    plt.scatter(df_model[var], fitted, color='red', s=15, alpha=0.4, label='Прогнозы P(approve=1)')
    
    # Настройки осей и легенды
    plt.ylim(-0.1, 1.1)
    plt.xlabel(x_vars[i])
    plt.ylabel('approve')
    
    # ВЕРНУЛИ ЗАГОЛОВОК ГРАФИКА
    plt.title(f'Зависимость LFP от {x_vars[i]}')
    
    plt.legend(loc='center right') 
    plt.tight_layout()
    plt.show()

**Доля правильных прогнозов (Accuracy) на обучающей выборке:** `87.6%`


<br>

### Анализ переменной `appinc`

<Figure size 2400x1200 with 1 Axes>

<br>

### Анализ переменной `dep`

<Figure size 2400x1200 with 1 Axes>

<br>

### Анализ переменной `yjob`

<Figure size 2400x1200 with 1 Axes>

## labour force equation

Для датасета `TableF5-1` рассмотрим регрессию __LFP на WA, I(WA ** 2), WE, KL6, K618, CIT, UN, np.log(FAMINC)__

In [74]:
#| echo: false
#| warning: false
#| message: false
#| output: asis

# 1. Строим модель
res_hc = ols(formula='LFP ~ 1 + WA + I(WA ** 2) + WE + KL6 + K618 + CIT + UN + np.log(FAMINC)', data=mroz_Greene).fit(cov_type='HC3')

# 2. Извлекаем данные 
actual = res_hc.model.endog
fitted = res_hc.fittedvalues

# Делаем .copy(), чтобы pandas не ругался на изменение исходного датасета
df_model = res_hc.model.data.frame.copy() 

# СОЗДАЕМ КОЛОНКУ С ЛОГАРИФМОМ ВРУЧНУЮ для графика
df_model['log_FAMINC'] = np.log(df_model['FAMINC'])

# 3. Считаем предсказательную силу (Accuracy)
threshold = 0.5
predicted_classes = (fitted >= threshold).astype(int)
accuracy = (predicted_classes == actual).mean()

display(Markdown(f"**Доля правильных прогнозов (Accuracy) на обучающей выборке:** `{accuracy:.1%}`\n"))

# ВАЖНО: теперь мы ссылаемся на нашу новую колонку 'log_FAMINC'
x_vars = ['WA', 'log_FAMINC', 'UN']
titles = ['WA', 'log(FAMINC)', 'UN']

# 4. Строим графики в цикле

for i, var in enumerate(x_vars):
    # Текстовое пояснение перед картинкой (по желанию можно убрать, если хватает title)
    display(Markdown(f"<br>\n\n### Анализ переменной `{titles[i]}`"))
    
    # Создаем новую отдельную фигуру
    plt.figure(figsize=(8, 4))
    
    # Рисуем
    plt.scatter(df_model[var], actual, color='blue', s=15, alpha=0.4, label='Фактические (LFP)')
    plt.scatter(df_model[var], fitted, color='red', s=15, alpha=0.4, label='Прогнозы P(LFP=1)')
    
    # Настройки осей и легенды
    plt.ylim(-0.1, 1.1)
    plt.xlabel(titles[i])
    plt.ylabel('LFP')
    
    # ВЕРНУЛИ ЗАГОЛОВОК ГРАФИКА
    plt.title(f'Зависимость LFP от {titles[i]}')
    
    plt.legend(loc='center right') 
    plt.tight_layout()
    plt.show()

**Доля правильных прогнозов (Accuracy) на обучающей выборке:** `66.4%`


<br>

### Анализ переменной `WA`

<Figure size 2400x1200 with 1 Axes>

<br>

### Анализ переменной `log(FAMINC)`

<Figure size 2400x1200 with 1 Axes>

<br>

### Анализ переменной `UN`

<Figure size 2400x1200 with 1 Axes>